# 🦘 TPUKangaroo — Bitcoin Puzzle Solver (Jacobian Edition)

**Implementação JAX/XLA otimizada** para Google Colab TPU v5e-1 (Python 3).

### Melhorias vs versão original:
- ✅ **Coordenadas Jacobianas** — ZERO inversões modulares no loop principal (~50–100× mais rápido)
- ✅ **Mega-loop** — centenas de saltos por sincronização host↔TPU
- ✅ **I/O assíncrono** — escrita de DPs em background thread
- ✅ **Inversão em lote (Montgomery)** — só na extração de DPs (evento raro)

---
**Ambiente recomendado**: Runtime → Alterar tipo de ambiente de execução → **TPU v5e-1**

In [ ]:
# ── Célula 1: Verificar ambiente TPU ─────────────────────────────────────────
import subprocess, sys

print('Python:', sys.version)

# Verificar se temos TPU disponível
try:
    import jax
    print('JAX version:', jax.__version__)
    print('Devices:', jax.devices())
    device_kind = str(jax.devices()[0]).lower()
    if 'tpu' in device_kind:
        print('✅ TPU detectado!')
        BACKEND = 'tpu'
    elif 'gpu' in device_kind:
        print('✅ GPU detectada!')
        BACKEND = 'gpu'
    else:
        print('⚠️  Apenas CPU disponível — performance limitada')
        BACKEND = 'cpu'
except ImportError:
    print('JAX não instalado, instalando...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'jax[tpu]',
                    '-f', 'https://storage.googleapis.com/jax-releases/libtpu_releases.html'])
    import jax
    BACKEND = 'tpu'
    print('JAX instalado:', jax.__version__)

In [ ]:
# ── Célula 2: Clonar repositório ──────────────────────────────────────────────
import os

if not os.path.exists('/content/TPUKangaroo'):
    !git clone https://github.com/ggsofthouse/TPUKangaroo.git /content/TPUKangaroo

%cd /content/TPUKangaroo
!git pull origin main
print('✅ Repositório pronto')

In [ ]:
# ── Célula 3: Configuração do Puzzle ─────────────────────────────────────────
#
# 🎯 Edite aqui para escolher o puzzle alvo
# Puzzles disponíveis: 20, 30, 40, 50, 60, 70, 80, 90, 100

PUZZLE_NUM       = 80          # Número do puzzle Bitcoin
KANGAROOS        = 131072      # Número total de kangaroos (tame + wild)
STEPS_PER_BLOCK  = 50          # Passos do fori_loop interno por bloco
N_BLOCKS         = 20          # Blocos por chamada ao TPU (sem sync host)
DP_BITS          = None        # None = automático (recomendado)
JUMP_TABLE_SIZE  = 64          # Tamanho da tabela de saltos

# TOTAL de passos por sincronização = STEPS_PER_BLOCK × N_BLOCKS
print(f'📊 Config:')
print(f'   Puzzle #{PUZZLE_NUM}')
print(f'   Backend: {BACKEND}')
print(f'   Kangaroos: {KANGAROOS:,}')
print(f'   Steps/sync: {STEPS_PER_BLOCK * N_BLOCKS:,}')

from puzzles_config import PUZZLES, print_colab_command
if PUZZLE_NUM in PUZZLES:
    p = PUZZLES[PUZZLE_NUM]
    PUBKEY = p['pubkey']
    START  = p['start']
    RANGE  = p['range']
    print(f'   Pubkey: {PUBKEY[:24]}...')
    print(f'   Start: 0x{START}')
    print(f'   Range: {RANGE} bits')
else:
    raise ValueError(f'Puzzle #{PUZZLE_NUM} não encontrado!')

In [ ]:
# ── Célula 4: Executar o solver ───────────────────────────────────────────────
#
# Monta e executa o comando do solver.
# A saída aparece inline no notebook.

import subprocess, sys, os

cmd = [
    sys.executable,
    'jax_kangaroo_tpu.py',
    '--backend',          BACKEND,
    '--range',            str(RANGE),
    '--start',            START,
    '--pubkey',           PUBKEY,
    '--kangaroos',        str(KANGAROOS),
    '--steps-per-block',  str(STEPS_PER_BLOCK),
    '--n-blocks',         str(N_BLOCKS),
    '--jump-table-size',  str(JUMP_TABLE_SIZE),
    '--steps',            '0',   # 0 = rodar indefinidamente
]

if DP_BITS is not None:
    cmd += ['--dp-bits', str(DP_BITS)]

print('⚙️  Comando:', ' '.join(cmd))
print('=' * 70)

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd='/content/TPUKangaroo'
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
print(f'\n✅ Processo finalizado com código {proc.returncode}')

In [ ]:
# ── Célula 5 (Opcional): Verificar RESULTS.TXT ───────────────────────────────
import os

results_file = '/content/TPUKangaroo/RESULTS.TXT'
if os.path.exists(results_file):
    with open(results_file) as f:
        content = f.read().strip()
    if content:
        print('🏆 RESULTADOS ENCONTRADOS:')
        print('=' * 60)
        print(content)
    else:
        print('ℹ️  RESULTS.TXT existe mas está vazio.')
else:
    print('ℹ️  RESULTS.TXT não encontrado ainda.')

In [ ]:
# ── Célula 6 (Opcional): Benchmark de velocidade ──────────────────────────────
#
# Roda por 60 segundos e reporta a velocidade do TPU sem buscar puzzle real.

import subprocess, sys

cmd_bench = [
    sys.executable,
    'jax_kangaroo_tpu.py',
    '--backend',         BACKEND,
    '--range',           str(RANGE),
    '--kangaroos',       str(KANGAROOS),
    '--steps-per-block', str(STEPS_PER_BLOCK),
    '--n-blocks',        str(N_BLOCKS),
    '--steps',           str(STEPS_PER_BLOCK * N_BLOCKS * 10),  # 10 chamadas e para
]

print('🔥 Rodando benchmark (10 mega-loop calls)...')
proc = subprocess.Popen(cmd_bench, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, cwd='/content/TPUKangaroo')
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

## 📊 Guia de Performance Esperada

| Puzzle | Bits | Ops Necessárias (K=1.15) | TPU v5e-1 Jacobian | RTX 4090 (ref) |
|--------|------|--------------------------|---------------------|----------------|
| **#80** | 79 | ~1.1T | **~5–15 min** | 76s |
| **#90** | 89 | ~37T | **~3–9 horas** | 43 min |
| **#100** | 99 | ~1.18P | **~4–12 dias** | 23 horas |

> ⚠️ As estimativas de TPU assumem **200 Mops/s com Jacobian** (vs 50 Mops/s affine).
> O benchmark real pode variar bastante dependendo do tamanho do batch e configuração do Colab.

## 🔧 Parâmetros de Tuning

| Parâmetro | Puzzle #80 | Puzzle #90 | Puzzle #100 |
|-----------|-----------|-----------|------------|
| `KANGAROOS` | 131072 | 262144 | 524288 |
| `STEPS_PER_BLOCK` | 50 | 100 | 200 |
| `N_BLOCKS` | 20 | 10 | 5 |
| `DP_BITS` | auto | auto | auto |